# 01 — Data Audit

Inventory all PDF files in the two source directories, build a preliminary metadata table, and surface any quality issues (duplicates, unexpected naming patterns, missing pages).

**Run this notebook before any other pipeline step.**

In [ ]:
# Setup: imports and project root
import sys
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Add project root to sys.path so src/ imports work
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
RENDERED_DIR = DATA_DIR / 'rendered_pages'
SPLITS_DIR = DATA_DIR / 'splits'

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Data dir exists: {DATA_DIR.exists()}')

## 1. Discover PDFs

Scan both source subdirectories (`pdf_s - שאלונים וכתב יד` and `מסמכים רגילים`) recursively for all `.pdf` files.

In [ ]:
# Discover all PDF files across source subdirectories
# Exclude output dirs (rendered_pages, splits)
EXCLUDE_DIRS = {'rendered_pages', 'splits'}

subdirs = [
    d for d in DATA_DIR.iterdir()
    if d.is_dir() and d.name not in EXCLUDE_DIRS
]
print(f'Found {len(subdirs)} source subdirectory(ies):')
for s in subdirs:
    print(f'  {s.name}')

all_pdfs = []
for subdir in subdirs:
    pdfs = list(subdir.rglob('*.pdf'))
    all_pdfs.extend(pdfs)
    print(f'  [{subdir.name}] -> {len(pdfs)} PDFs')

print(f'\nTotal PDFs discovered: {len(all_pdfs)}')

## 2. Build Preliminary Metadata

Construct a DataFrame with one row per PDF file. Columns:
- `file_path` — path relative to `data/` (used as the primary key)
- `source_dir` — which subdirectory the file came from
- `filename` — bare filename (stem + suffix)
- `stem` — filename without extension
- `page_suffix` — extracted `_page_XXXX` token if present

In [ ]:
# Build metadata DataFrame from discovered PDF paths
PAGE_PATTERN = re.compile(r'_page_(\d+)', re.IGNORECASE)

records = []
for pdf_path in all_pdfs:
    # Relative path from data/ — used as stable key across pipeline steps
    rel_path = pdf_path.relative_to(DATA_DIR)
    stem = pdf_path.stem
    match = PAGE_PATTERN.search(stem)
    page_suffix = match.group(0) if match else None
    page_num = int(match.group(1)) if match else None

    records.append({
        'file_path': str(rel_path),
        'source_dir': pdf_path.parent.name,
        'filename': pdf_path.name,
        'stem': stem,
        'page_suffix': page_suffix,
        'page_num': page_num,
        'abs_path': str(pdf_path),  # kept locally for rendering; not saved to CSV
    })

meta_df = pd.DataFrame(records)
print(f'metadata shape: {meta_df.shape}')
meta_df.head(10)

## 3. Distribution Statistics

In [ ]:
# Per-source-directory count and percentage
if meta_df.empty:
    print('No PDFs found — check that data subdirectories contain .pdf files.')
else:
    counts = meta_df['source_dir'].value_counts()
    pct = (counts / len(meta_df) * 100).round(1)
    dist = pd.DataFrame({'count': counts, 'pct': pct})
    print('PDFs per source directory:')
    print(dist.to_string())
    print(f'\nTotal: {len(meta_df)}')

    # Page number distribution (only for files with a _page_XXXX suffix)
    has_page = meta_df['page_num'].notna()
    print(f'\nFiles with _page_XXXX pattern: {has_page.sum()} / {len(meta_df)}')
    if has_page.sum() > 0:
        page_nums = meta_df.loc[has_page, 'page_num']
        print(f'page_num range: {int(page_nums.min())} – {int(page_nums.max())}')
        print(f'Unique page numbers: {page_nums.nunique()}')

    # Bar chart: PDFs per source directory
    fig, ax = plt.subplots(figsize=(7, 3))
    counts.plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452'], edgecolor='black')
    ax.set_title('PDF Count per Source Directory')
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

## 4. Filename Analysis

- Detect Hebrew characters in filenames (Unicode range `\u0590`–`\u05FF`)
- Verify the expected `_page_XXXX.pdf` suffix pattern
- Identify any files that deviate from the pattern

In [ ]:
# Filename analysis: Hebrew characters and page-suffix pattern
HEBREW_RE = re.compile(r'[\u0590-\u05FF]')

if meta_df.empty:
    print('No files to analyse.')
else:
    meta_df['has_hebrew'] = meta_df['filename'].apply(lambda n: bool(HEBREW_RE.search(n)))
    meta_df['has_page_pattern'] = meta_df['page_suffix'].notna()

    print(f'Files with Hebrew in filename : {meta_df["has_hebrew"].sum()} / {len(meta_df)}')
    print(f'Files with _page_XXXX pattern : {meta_df["has_page_pattern"].sum()} / {len(meta_df)}')

    # Show files WITHOUT the expected page pattern
    anomalous = meta_df[~meta_df['has_page_pattern']]
    if anomalous.empty:
        print('\nAll files follow the _page_XXXX pattern. ✓')
    else:
        print(f'\nFiles WITHOUT _page_XXXX pattern ({len(anomalous)}):')
        print(anomalous[['file_path', 'source_dir']].to_string(index=False))

    # Show unique page numbers present (sorted)
    if meta_df['page_num'].notna().any():
        unique_pages = sorted(meta_df['page_num'].dropna().astype(int).unique())
        print(f'\nUnique page numbers present: {unique_pages[:30]}'
              f'{" ..." if len(unique_pages) > 30 else ""}')

## 5. Duplicate Check

Look for duplicate filenames within each subdirectory and across subdirectories.

In [ ]:
# Duplicate detection: within-subdir and cross-subdir
if meta_df.empty:
    print('No files to check.')
else:
    # Within-subdir duplicates
    within_dupes = meta_df[meta_df.duplicated(subset=['source_dir', 'filename'], keep=False)]
    if within_dupes.empty:
        print('No duplicate filenames within any subdirectory. ✓')
    else:
        print(f'Within-subdir duplicates ({len(within_dupes)} rows):')
        print(within_dupes[['source_dir', 'filename', 'file_path']].to_string(index=False))

    # Cross-subdir duplicates (same filename in both source dirs)
    cross_dupes = meta_df[meta_df.duplicated(subset=['filename'], keep=False)]
    cross_cross = cross_dupes[cross_dupes.groupby('filename')['source_dir'].transform('nunique') > 1]
    if cross_cross.empty:
        print('No cross-directory duplicate filenames. ✓')
    else:
        print(f'\nCross-directory duplicates ({len(cross_cross)} rows):')
        print(cross_cross[['source_dir', 'filename', 'file_path']].sort_values('filename').to_string(index=False))

## 6. Sample Display

In [ ]:
# Show first 10 rows of the metadata table (display columns useful for annotation)
display_cols = ['file_path', 'source_dir', 'filename', 'page_num', 'has_hebrew', 'has_page_pattern']
available_cols = [c for c in display_cols if c in meta_df.columns]

if meta_df.empty:
    print('No metadata to display.')
else:
    print('First 10 rows of preliminary metadata:')
    display(meta_df[available_cols].head(10))

## 7. Audit Summary

| Check | Result |
|---|---|
| Total PDFs found | (see cell output above) |
| Source directories | 2 expected (`pdf_s - שאלונים וכתב יד`, `מסמכים רגילים`) |
| Files with Hebrew filenames | reported above |
| Files following `_page_XXXX` pattern | reported above |
| Within-subdir duplicates | reported above |
| Cross-directory duplicates | reported above |

### Next Steps
1. If any files are missing the `_page_XXXX` suffix, confirm whether they are single-page originals or incorrectly named.
2. Resolve any duplicate filenames before annotation begins.
3. Proceed to `02_rendering_checks.ipynb` to verify PDF → PNG rendering quality.
4. After annotation, run `03_label_consistency.ipynb` to validate label alignment.